In [ ]:
import pandas as pd

# ==============================================================================
# DATA SETUP (Pure Python + Pandas)
# ==============================================================================
raw_records = {
    "student_id": ["ST_01", "ST_02", "ST_03", "ST_04", "ST_05", "ST_06", "ST_01"],  # Duplicate row
    "course": ["AI", "NLP", "AI", "Python", "NLP", "AI", "AI"],
    "hours": [12, 25, 18, 8, 30, 22, 12],
    "loss": [0.45, 0.20, 0.65, 1.10, 0.15, 0.32, 0.45],
    "score": [88.0, 94.0, None, 62.0, 98.0, 91.0, 88.0],                             # Missing value
    "status": ["PASS", "PASS", "FAIL", "FAIL", "PASS", "PASS", "PASS"]
}
df = pd.DataFrame(raw_records)

# ==============================================================================
# PROGRESSIVE PIPELINE: 20 CORE PANDAS METHODS (Simple -> Advanced)
# ==============================================================================

# --- Phase 1: Inspection (Simple) ---
# 1. head(): View the first N records
print("1. head():\n", df.head(2), "\n")

# 2. info(): Memory footprint, non-null counts, and datatypes
print("2. info():")
df.info()
print()

# 3. describe(): Summary statistics of numeric attributes
print("3. describe():\n", df.describe(), "\n")

# 4. value_counts(): Frequency distribution of categories
print("4. value_counts():\n", df["course"].value_counts(), "\n")

# --- Phase 2: Selection & Filtering ---
# 5. loc[]: Label-based & boolean condition indexing
print("5. loc[]:\n", df.loc[df["hours"] >= 20, ["student_id", "course", "hours"]], "\n")

# 6. iloc[]: Positional integer-based indexing (rows 0-2, columns 0-3)
print("6. iloc[]:\n", df.iloc[0:3, 0:3], "\n")

# 7. isin(): Boolean mask over membership criteria
print("7. isin():\n", df[df["course"].isin(["AI", "NLP"])], "\n")

# 8. query(): Expression-based conditional filtering
min_hours = 15
print("8. query():\n", df.query("hours > @min_hours and status == 'PASS'"), "\n")

# --- Phase 3: Data Cleaning & Sanitization ---
# 9. isna(): Detect missing values per column
print("9. isna():\n", df.isna().sum(), "\n")

# 10. drop_duplicates(): Eliminate redundant records by key
df = df.drop_duplicates(subset=["student_id"], keep="first")
print(f"10. drop_duplicates(): Remaining records = {len(df)}\n")

# 11. fillna(): Impute missing values with statistical central tendency
df["score"] = df["score"].fillna(df["score"].mean())
print("11. fillna():\n", df[["student_id", "score"]], "\n")

# 12. drop(): Remove non-essential dimensions/columns
df_trimmed = df.drop(columns=["status"])
print("12. drop(): Remaining columns =", list(df_trimmed.columns), "\n")

# --- Phase 4: Transformation & Feature Engineering ---
# 13. astype(): Type casting for memory/categorical representation
df["course"] = df["course"].astype("category")
print("13. astype(): course dtype =", df["course"].dtype, "\n")

# 14. map(): Dictionary-based value substitution / categorical encoding
df["pass_flag"] = df["status"].map({"PASS": 1, "FAIL": 0})
print("14. map():\n", df[["status", "pass_flag"]], "\n")

# 15. assign(): Functional column creation via method chaining
df = df.assign(efficiency=lambda x: x["score"] / x["hours"])
print("15. assign():\n", df[["student_id", "score", "hours", "efficiency"]].head(3), "\n")

# 16. sort_values(): Sort dataframe by single or multiple keys
print("16. sort_values():\n", df.sort_values(by="score", ascending=False).head(3), "\n")

# --- Phase 5: Grouping, Aggregation & Reshaping (Advanced) ---
# 17. groupby(): Split-Apply-Combine engine
print("17. groupby():\n", df.groupby("course", observed=True)["score"].mean(), "\n")

# 18. agg(): Multi-metric multi-column aggregation
grouped_metrics = df.groupby("course", observed=True).agg(
    mean_score=("score", "mean"),
    max_hours=("hours", "max"),
    sample_count=("student_id", "count")
)
print("18. agg():\n", grouped_metrics, "\n")

# 19. reset_index(): Flatten grouped index back into regular columns
tidy_metrics = grouped_metrics.reset_index()
print("19. reset_index():\n", tidy_metrics, "\n")

# 20. merge(): Relational join with external metadata table
course_info = pd.DataFrame({
    "course": ["AI", "NLP", "Python"],
    "level": ["Advanced", "Advanced", "Beginner"]
})
final_merged = pd.merge(df, course_info, on="course", how="left")
print("20. merge():\n", final_merged[["student_id", "course", "level", "score"]].head(4))


1. head():
   student_id course  hours  loss  score status
0      ST_01     AI     12  0.45   88.0   PASS
1      ST_02    NLP     25  0.20   94.0   PASS 

2. info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   student_id  7 non-null      object 
 1   course      7 non-null      object 
 2   hours       7 non-null      int64  
 3   loss        7 non-null      float64
 4   score       6 non-null      float64
 5   status      7 non-null      object 
dtypes: float64(2), int64(1), object(3)
memory usage: 464.0+ bytes

3. describe():
            hours      loss      score
count   7.000000  7.000000   6.000000
mean   18.142857  0.474286  86.833333
std     7.967195  0.323463  12.750163
min     8.000000  0.150000  62.000000
25%    12.000000  0.260000  88.000000
50%    18.000000  0.450000  89.500000
75%    23.500000  0.550000  93.250000
max    30.000000  

In [ ]:
import pandas as pd

# ==============================================================================
# SCENARIO: Valence Academy - Student Capstone Model Evaluation Pipeline
# We evaluate student AI capstones: inspecting logs, cleaning corrupted runs,
# computing performance metrics, and linking instructor metadata for graduation.
# ==============================================================================

# ------------------------------------------------------------------------------
# 0. INGESTION: Raw submission logs with dirty entries (duplicates, NaNs)
# ------------------------------------------------------------------------------
raw_submissions = {
    "student_id": ["ST_101", "ST_102", "ST_103", "ST_104", "ST_105", "ST_106", "ST_101"],  # Duplicate run
    "track": ["Vision", "NLP", "Vision", "Tabular", "NLP", "Vision", "Vision"],
    "model": ["ResNet50", "BERT", "YOLOv8", "XGBoost", "RoBERTa", "ResNet50", "ResNet50"],
    "epochs": [25, 40, 30, 10, 50, 35, 25],
    "val_loss": [0.38, 0.19, 0.72, 0.85, 0.12, 0.28, 0.38],
    "f1_score": [0.89, 0.95, None, 0.64, 0.98, 0.91, 0.89],                                 # Missing metric
    "eval_status": ["PASSED", "PASSED", "FAILED", "FAILED", "PASSED", "PASSED", "PASSED"]
}
df = pd.DataFrame(raw_submissions)

# ==============================================================================
# 20 ESSENTIAL PANDAS METHODS (Progressive Operational Pipeline)
# ==============================================================================

# ------------------------------------------------------------------------------
# Phase 1: Rapid Triage & Sanity Checks (Methods 1-4)
# ------------------------------------------------------------------------------
# 1. head(): First look at incoming batch of submissions
print("1. head() -> Quick sanity check of incoming batch:")
print(df.head(3), "\n")

# 2. info(): Check dtypes, total records, and locate missing values
print("2. info() -> Structural audit:")
df.info()
print()

# 3. describe(): Statistical dispersion across training runs
print("3. describe() -> Distribution of training metrics:")
print(df.describe(), "\n")

# 4. value_counts(): Distribution of submissions across learning tracks
print("4. value_counts() -> Capstone submissions per track:")
print(df["track"].value_counts(), "\n")

# ------------------------------------------------------------------------------
# Phase 2: Targeted Querying & Slice Extraction (Methods 5-8)
# ------------------------------------------------------------------------------
# 5. loc[]: Filter top-tier submissions by label and conditional threshold
print("5. loc[] -> High-performing runs (loss < 0.30):")
print(df.loc[df["val_loss"] < 0.30, ["student_id", "model", "val_loss"]], "\n")

# 6. iloc[]: Positional coordinate extraction (first 3 submissions, first 4 attributes)
print("6. iloc[] -> Array coordinate slice [0:3, 0:4]:")
print(df.iloc[0:3, 0:4], "\n")

# 7. isin(): Isolate specific deep learning architectures
print("7. isin() -> Filter specific neural architectures (ResNet50 & BERT):")
print(df[df["model"].isin(["ResNet50", "BERT"])], "\n")

# 8. query(): String expression query for robust candidate validation
min_epochs = 20
print("8. query() -> Filter qualified runs via parameterized expression:")
print(df.query("epochs >= @min_epochs and eval_status == 'PASSED'"), "\n")

# ------------------------------------------------------------------------------
# Phase 3: Data Cleaning & Hygiene (Methods 9-12)
# ------------------------------------------------------------------------------
# 9. isna(): Detect exact null counts before running downstream calculations
print("9. isna() -> Missing value count across columns:")
print(df.isna().sum(), "\n")

# 10. drop_duplicates(): Remove redundant re-runs from the same student
df = df.drop_duplicates(subset=["student_id"], keep="first")
print(f"10. drop_duplicates() -> Cleaned unique students: {len(df)} records\n")

# 11. fillna(): Impute missing benchmark score using the cohort median
median_f1 = df["f1_score"].median()
df["f1_score"] = df["f1_score"].fillna(median_f1)
print("11. fillna() -> Imputed missing F1 scores with cohort median:")
print(df[["student_id", "f1_score"]], "\n")

# 12. drop(): Remove deprecated submission flags
df_clean = df.drop(columns=["eval_status"])
print("12. drop() -> Retained analytical schema:")
print(df_clean.columns.tolist(), "\n")

# ------------------------------------------------------------------------------
# Phase 4: Feature Engineering & Preprocessing (Methods 13-16)
# ------------------------------------------------------------------------------
# 13. astype(): Downcast strings to categories to reduce footprint & enforce enum semantics
df["track"] = df["track"].astype("category")
print(f"13. astype() -> 'track' dtype converted to: {df['track'].dtype}\n")

# 14. map(): Encode binary pass/fail labels into numerical flags
df["certified_flag"] = df["eval_status"].map({"PASSED": 1, "FAILED": 0})
print("14. map() -> Binary label encoding:")
print(df[["student_id", "eval_status", "certified_flag"]], "\n")

# 15. assign(): Derive training efficiency (F1 gain per training epoch)
df = df.assign(f1_per_epoch=lambda x: x["f1_score"] / x["epochs"])
print("15. assign() -> Derived metric (f1_per_epoch):")
print(df[["student_id", "epochs", "f1_score", "f1_per_epoch"]].head(3), "\n")

# 16. sort_values(): Rank students from best to worst based on validation loss
print("16. sort_values() -> Top performing models ranked by lowest loss:")
print(df.sort_values(by="val_loss", ascending=True)[["student_id", "model", "val_loss"]], "\n")

# ------------------------------------------------------------------------------
# Phase 5: Multi-dimensional Aggregation & Relational Joins (Methods 17-20)
# ------------------------------------------------------------------------------
# 17. groupby(): Group metrics across different academy tracks
print("17. groupby() -> Mean validation loss per track:")
print(df.groupby("track", observed=True)["val_loss"].mean(), "\n")

# 18. agg(): Produce an executive track-level scoreboard
track_scoreboard = df.groupby("track", observed=True).agg(
    avg_f1=("f1_score", "mean"),
    min_loss=("val_loss", "min"),
    student_count=("student_id", "count")
)
print("18. agg() -> Multi-metric performance scorecard:")
print(track_scoreboard, "\n")

# 19. reset_index(): Flatten grouped multi-index scorecard into a standard DataFrame
final_scoreboard = track_scoreboard.reset_index()
print("19. reset_index() -> Flattened tabular layout:")
print(final_scoreboard, "\n")

# 20. merge(): Relational join connecting track leads to student performance
faculty_registry = pd.DataFrame({
    "track": ["Vision", "NLP", "Tabular"],
    "mentor_lead": ["Dr. Mehrshad", "Dr. Sara", "Eng. Reza"]
})
graduating_roster = pd.merge(df, faculty_registry, on="track", how="left")
print("20. merge() -> Enriched student records with assigned mentor leads:")
print(graduating_roster[["student_id", "track", "mentor_lead", "f1_score", "certified_flag"]])
